In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
import os

In [78]:
# Duomenų rinkiniai ir klasterių skaičiai pagal:
# Empirinis, Alkūnės, Vidutinio silueto metodus
cluster_counts = {
    "norm_full.csv": [76, 6, 3],
    "norm_6.csv": [27, 7, 13],
    "dvi_dim.csv": [27, 10, 11]
}
cluster_labels = ["empirinis", "alkunes", "silueto"]

base_folder = "hierarchical"  # Pagrindinis aplankas rezultatams
linkage_method = "ward"       # Naudojamas sujungimo metodas
metric_type = "euclidean"     # Naudojama metrika


In [79]:
for file in cluster_counts.keys():
    print(f"\n🔹 Apdorojamas duomenų failas: {file}")
    df = pd.read_csv(file)

    if 'label' in df.columns:
        X = df.drop(columns=['label']).values
    else:
        X = df.values

    print(f"Duomenys nuskaityti: {file}")

    # Sukuriamas atitinkamas aplankas
    file_folder = os.path.join(base_folder, file[:-4])
    os.makedirs(file_folder, exist_ok=True)

    # t-SNE nustatymai
    if file == "dvi_dim.csv":
        X_tsne = np.array(X)
    elif file == "norm_full.csv":
        perp, learn_r, early_ex = 50, 200, 24
    elif file == "norm_6.csv":
        perp, learn_r, early_ex = 50, 50, 12

    if file != "dvi_dim.csv":
        tsne = TSNE(
            n_components=2,         # sumažiname duomenų dimensijų skaičių iki 2
            perplexity=perp,        # kiek artimiausių kaimynų laikoma reikšmingais
            learning_rate=learn_r,  # mokymosi greitis
            max_iter=1000,          # iteracijų skaičius
            early_exaggeration=early_ex,  # padeda atskirti klasterius
            metric="euclidean",     # atstumo metrika
            random_state=67,        # atkuriamumas
            init="pca"              # pradinė taškų pozicija nustatoma PCA pagrindu
        )
        X_tsne = tsne.fit_transform(X)

    # Dendograma
    print("   ➤ Kuriama dendrograma...")
    linked = linkage(X, method=linkage_method, metric=metric_type)

    plt.figure(figsize=(12, 6))
    dendrogram(
        linked,
        orientation='top',
        distance_sort='descending',
        show_leaf_counts=False
    )
    plt.title(f"{file} — Hierarchinio klasterizavimo dendrograma\n({linkage_method.capitalize()} + {metric_type.capitalize()})")
    plt.xlabel("Duomenų taškai")
    plt.ylabel("Atstumas")
    plt.tight_layout()
    plt.savefig(os.path.join(file_folder, f"dendrogram_{file[:-4]}.png"), dpi=200)
    plt.close()

    # t-SNE vizualizacijos kiekvienam klasterių skaičiui
    print("   ➤ Kuriami t-SNE grafikai...")
    for label, k in zip(cluster_labels, cluster_counts[file]):
        print(f"      ↳ {label} metodas (k={k})")

        agg = AgglomerativeClustering(
            n_clusters=k,
            linkage=linkage_method,
            metric=metric_type
        )
        y_pred = agg.fit_predict(X)

        # Vizualizacija
        plt.figure(figsize=(9, 9))
        cmap = ListedColormap(
            list(plt.cm.tab20.colors) +
            list(plt.cm.tab20b.colors) +
            list(plt.cm.tab20c.colors)
        )
        plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_pred, cmap=cmap, s=50)
        plt.title(f"{file} — t-SNE pagal hierarchinius klasterius\n({label}, k={k})")
        plt.xlabel("t-SNE komponentė 1")
        plt.ylabel("t-SNE komponentė 2")
        plt.grid(True)
        plt.tight_layout()

        tsne_path = os.path.join(file_folder, f"tsne_{file[:-4]}_{label}_{k}.png")
        plt.savefig(tsne_path, dpi=200)
        plt.close()

    print(f"Grafikai išsaugoti aplanke: {file_folder}")

print("\n Duomenų rinkiniai apdoroti ir išsaugoti.")


🔹 Apdorojamas duomenų failas: norm_full.csv
Duomenys nuskaityti: norm_full.csv
   ➤ Kuriama dendrograma...
   ➤ Kuriami t-SNE grafikai...
      ↳ empirinis metodas (k=76)
      ↳ alkunes metodas (k=6)
      ↳ silueto metodas (k=3)
Grafikai išsaugoti aplanke: hierarchical\norm_full

🔹 Apdorojamas duomenų failas: norm_6.csv
Duomenys nuskaityti: norm_6.csv
   ➤ Kuriama dendrograma...
   ➤ Kuriami t-SNE grafikai...
      ↳ empirinis metodas (k=27)
      ↳ alkunes metodas (k=7)
      ↳ silueto metodas (k=13)
Grafikai išsaugoti aplanke: hierarchical\norm_6

🔹 Apdorojamas duomenų failas: dvi_dim.csv
Duomenys nuskaityti: dvi_dim.csv
   ➤ Kuriama dendrograma...
   ➤ Kuriami t-SNE grafikai...
      ↳ empirinis metodas (k=27)
      ↳ alkunes metodas (k=10)
      ↳ silueto metodas (k=11)
Grafikai išsaugoti aplanke: hierarchical\dvi_dim

 Duomenų rinkiniai apdoroti ir išsaugoti.


In [80]:
# df = pd.read_csv(file)
# X = df.drop(columns=['label']).values if 'label' in df.columns else df.values
# print(f"✅ Data loaded: {file}, shape = {X.shape}")
#
# # === t-SNE setup ===
# if file == "dvi_dim.csv":
#     X_tsne = np.array(X)
# elif file == "norm_full.csv":
#     perp, learn_r, early_ex = 50, 200, 24
# elif file == "norm_6.csv":
#     perp, learn_r, early_ex = 50, 50, 12
#
# if file != "dvi_dim.csv":
#     tsne = TSNE(
#         n_components=2,         # sumažiname duomenų dimensijų skaičių iki 2
#         perplexity=perp,          # kiek artimiausių kaimynų laikoma reikšmingais
#         learning_rate=learn_r,      # nustato, kokio dydžio korekcijos taikomos kiekviename iteracijos žingsnyje
#         max_iter=1000,            # iteracijų skaičius: kiek gradientinio nusileidimo žingsnių atliekama minimizuojant klaidą
#         early_exaggeration=early_ex,# padeda atskirti klasterius ir išvengti lokalių minimumų
#         metric="euclidean",     # atstumo metrika pradinėje erdvėje
#         random_state=67,        # kad rezultatai būtų atkuriami (užtikrina tą pačią pradinę taškų padėtį)
#         init="pca"              # pradinė taškų pozicija nustatoma PCA pagrindu (stabiliau nei random)
#     )
#
#     X_tsne = tsne.fit_transform(X)

In [81]:
# for linkage_method in linkage_methods:
#     for metric_type in metrics:
#         # Ward apjungimas palaiko tik Euclidean metriką
#         if linkage_method == "ward" and metric_type != "euclidean":
#             continue
#
#         print(f"\n🔹 Processing {file} | {linkage_method.upper()} + {metric_type.upper()}")
#
#         # Failu strukturos apibrezimas
#         base_folder = "hierarchical"
#         file_folder = os.path.join(base_folder, file[:-4])
#         linkage_folder = os.path.join(file_folder, linkage_method)
#         os.makedirs(linkage_folder, exist_ok=True)
#
#         # Kiekvienas atskiras taskas laikomas savo atskiru klasteriu
#         agg = AgglomerativeClustering(
#             n_clusters=klasteriu_sk,
#             linkage=linkage_method,
#             metric=metric_type
#         )
#         y_pred = agg.fit_predict(X)
#
#         # t-SNE projekcijos
#         plt.figure(figsize=(9, 9))
#         cmap = ListedColormap(
#             list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors) + list(plt.cm.tab20c.colors)
#         )
#         plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_pred, cmap=cmap, s=50)
#         plt.title(f"{file} — t-SNE pagal hierarchinius klasterius\n({linkage_method}, {metric_type}, k={klasteriu_sk})")
#         plt.xlabel("t-SNE komponentė 1")
#         plt.ylabel("t-SNE komponentė 2")
#         plt.grid(True)
#         plt.tight_layout()
#
#         tsne_path = os.path.join(linkage_folder, f"tsne_{file[:-4]}_{linkage_method}_{metric_type}.png")
#         plt.savefig(tsne_path, dpi=200)
#         plt.close()
#
#         print(f"Issaugota: {linkage_folder}")
#
# print("\nVisi failai sukurti ir issaugoti.")

In [82]:
# plt.figure(figsize=(10, 10))
#
# colors = plt.cm.tab20.colors + plt.cm.tab20b.colors + plt.cm.tab20c.colors + plt.cm.Set1.colors
# cmap = ListedColormap(colors)
#
# plt.scatter(
#     X_tsne[:, 0],
#     X_tsne[:, 1],
#     c=y_agg,
#     cmap=cmap,
#     s=50
# )
# plt.title(f"t-SNE vizualizacija pagal hierarchinius klasterius ({klasteriu_sk})")
# plt.xlabel("t-SNE komponentė 1")
# plt.ylabel("t-SNE komponentė 2")
# plt.grid(True)
# plt.show()


In [83]:
# linked = linkage(X, method='ward')
#
# plt.figure(figsize=(12, 6))
# dendrogram(
#     linked,
#     orientation='top',
#     distance_sort='descending',
#     show_leaf_counts=False
# )
# plt.title(f"{file} — Hierarchinio klasterizavimo dendrograma")
# plt.xlabel("Duomenų taškai")
# plt.ylabel("Atstumas")
# plt.grid(True)
# plt.show()

In [84]:
# from scipy.cluster.hierarchy import dendrogram, linkage
# import matplotlib.pyplot as plt
# import pandas as pd
#
# for file in ["norm_full.csv", "norm_6.csv", "dvi_dim.csv"]:
#     df = pd.read_csv(file)
#     X = df.drop(columns=['label']).values if 'label' in df.columns else df.values
#
#     linked = linkage(X, method='ward', metric='euclidean')
#
#     plt.figure(figsize=(12, 6))
#     dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=False)
#     plt.title(f"{file} — Dendrograma (Ward + Euklidinė metrika)")
#     plt.xlabel("Duomenų taškai")
#     plt.ylabel("Atstumas")
#     plt.tight_layout()
#     plt.savefig(f"dendrogram_{file[:-4]}_ward_euclidean.png", dpi=200)
#     plt.close()
#
# print("Issaugotos 3 dendogramos, po viena kiekvienam duomenu rinkiniui")